# 03 · LLMs and Structured Output

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama. OpenAI cells run only if
> `OPENAI_API_KEY` is set in `03_CREWAI/.env`.

Every CrewAI agent needs a model, and every program that acts on an agent's
answer needs that answer in a predictable shape. This notebook covers CrewAI's
`LLM` class (how it reaches Ollama, OpenAI and others) and **structured task
output**: getting a validated Pydantic object instead of prose. You'll also see
exactly how CrewAI sends your schema to the model, which turns out to differ
from the other frameworks in a way that matters.

**You will learn**
- The `LLM` class and `provider/model` names, and how CrewAI reaches Ollama without LiteLLM
- Model settings, calling a model directly, and different models per agent
- `output_pydantic` and `output_json`: typed task results
- **How the schema reaches the model** in CrewAI, and why your field descriptions matter here
- Validation limits, and counting tokens correctly

**Prerequisites:** notebooks 01–02.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

from typing import Literal
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, LLM
from common import has_key

## 1 · Why models are pluggable

A crew shouldn't care which model is behind it. You might develop on a free local
model, run on a cloud model in production, and use a cheap model for simple
agents and a strong one for hard ones. CrewAI wraps every provider in one
**`LLM`** class, named with a `provider/model` string:

| Name | What CrewAI uses |
|---|---|
| `openai/gpt-4.1-mini` | its native OpenAI client |
| `anthropic/claude-…`, `gemini/…`, `azure/…`, `bedrock/…` | native clients (install the provider's extra) |
| `ollama/qwen3-coder:30b` | its native **OpenAI-compatible** client, pointed at Ollama |
| other providers | LiteLLM, if installed (`pip install "crewai[litellm]"`) |

Older CrewAI versions routed almost everything through LiteLLM. In 1.x the common
providers are native, which is why this section doesn't install LiteLLM at all.

## 2 · Connecting to Ollama

`common.get_llm()` builds the `LLM` for whichever provider `.env` selects:

In [ ]:
import inspect, common
print(inspect.getsource(common.get_llm))

In [ ]:
llm = get_llm()
print(type(llm).__name__, '→', llm.model)
print('direct call:', llm.call('In one sentence: what is a context window?'))

`llm.call(...)` sends a prompt straight to the model, with no agent or crew.
It's handy for quick checks and for small utility steps inside Flows (notebook 06).

**Model settings** are constructor arguments: `temperature`, `max_tokens`,
`timeout`, `top_p`, `seed`, `base_url`, `api_key`, and more. Two useful patterns:

In [ ]:
creative = get_llm(temperature=1.2)
precise = get_llm(temperature=0)
ask = 'Invent one product name for a stair-climbing delivery robot. Reply with the name only.'
print('temperature 1.2:', [creative.call(ask) for _ in range(3)])
print('temperature 0  :', [precise.call(ask) for _ in range(3)])

**Different models per agent.** `llm` is per agent, so a crew can mix a cheap,
fast model for simple roles with a strong one where quality matters:

```python
triage = Agent(role='Triage', ..., llm=get_llm('ollama'))     # cheap, fast
writer = Agent(role='Writer', ..., llm=get_llm('openai'))     # strong
```

Every configured provider, same agent definition:

In [ ]:
async def explain(model):
    a = Agent(role='Explainer', goal='Explain briefly', backstory='Concise.', llm=model)
    t = Task(description='What is a vector database?', expected_output='One sentence.', agent=a)
    return (await Crew(agents=[a], tasks=[t]).kickoff_async()).raw

print('ollama :', await explain(get_llm('ollama')))
print('openai :', await explain(get_llm('openai')) if has_key('openai') else '(skipped: no OPENAI_API_KEY in .env)')

## 3 · Structured output: typed task results

When code will act on a task's result, give the task an `output_pydantic`
model (or `output_json`). The crew's result then carries a validated object in
`result.pydantic` (and a dict in `result.json_dict`):

In [ ]:
class Ticket(BaseModel):
    category: Literal['billing', 'technical', 'other']
    urgency: int = Field(ge=1, le=5, description=(
        '1 = question, no impact; 3 = something is broken; '
        '5 = customer is losing money now or faces a deadline within 48 hours'))
    summary: str


triager = Agent(role='Support triage', goal='Classify tickets accurately', backstory='Precise.', llm=llm)
classify = Task(description='Classify this support ticket: {ticket}', expected_output='A classified ticket',
                agent=triager, output_pydantic=Ticket)

result = await Crew(agents=[triager], tasks=[classify]).kickoff_async(
    inputs={'ticket': 'I was charged three times this month and my rent is due tomorrow!!'})
ticket = result.pydantic
print(type(ticket).__name__, ticket)
print('route on it:', 'escalate' if ticket.urgency >= 4 else 'queue', '→', ticket.category)

## 4 · How the schema reaches the model, and why it matters

Notice the urgency scale lives **only** in the field's `description`. In the
LangGraph and OpenAI Agents SDK sections, that exact setup failed on Ollama: the
schema was sent as a *decoding constraint* that shaped the JSON, but the model
never read the descriptions, so it guessed low urgencies.

CrewAI does it differently. Let's look at the actual prompt (with the prompt
spy from notebook 02):

In [ ]:
import asyncio
from crewai.events import BaseEventListener
from crewai.events.types.llm_events import LLMCallStartedEvent

captured = []
class PromptSpy(BaseEventListener):
    def setup_listeners(self, bus):
        @bus.on(LLMCallStartedEvent)
        def on_call(source, event):
            captured.append(event.messages)
spy = PromptSpy()

captured.clear()
await Crew(agents=[triager], tasks=[classify]).kickoff_async(inputs={'ticket': 'The app crashes when I log in.'})
await asyncio.sleep(0.3)
user_prompt = captured[0][-1]['content']
start = user_prompt.find('"urgency"')
print('…' + user_prompt[start:start + 330] + '…')

CrewAI writes the **JSON schema, descriptions included, into the prompt text**.
The model *reads* the urgency scale, which is why it chose 5 above. In our
testing it did so on every run (3 of 3), with the scale only in the description.

The general lesson is more important than the CrewAI detail: **"structured output"
means different things in different frameworks and providers.** It can be a
decoding constraint the model never reads, a schema in the prompt, or a tool
definition. Whether your field descriptions matter depends on which one. Test
it, or put decision rules in the task description, which works everywhere.

## 5 · `output_json`, and when validation fails

`output_json=Model` gives you a plain dict (`result.json_dict`) instead of an
object. Useful when the result goes straight into JSON (an API response, a file):

In [ ]:
class Plan(BaseModel):
    steps: list[str]
    owner: str

planner = Agent(role='Planner', goal='Make short plans', backstory='Practical.', llm=llm)
plan_task = Task(description='Plan a pilot of delivery robots in one city.', expected_output='A short plan',
                 agent=planner, output_json=Plan)
result = await Crew(agents=[planner], tasks=[plan_task]).kickoff_async()
print(type(result.json_dict).__name__, result.json_dict)

**Validation.** The object is validated by Pydantic, so `Literal` values,
`ge`/`le` bounds and required fields are guaranteed when you get one. If the
model's output can't be converted, CrewAI tries to repair it with a conversion
step before giving up. Keep schemas **small and flat** for local models:
deeply nested schemas fail more often.

And as always: **valid is not the same as correct.** A classifier that returns
urgency 3 for everything produces perfectly valid tickets. Test on known cases.

## 6 · Counting tokens, correctly

`result.token_usage` reports tokens and model calls, but remember notebook 02:
it's **accumulated on the LLM object**. To measure one run, use a fresh LLM
object (or subtract the value from before the run):

In [ ]:
fresh = get_llm()
a = Agent(role='Explainer', goal='Explain', backstory='Concise.', llm=fresh)
out = await Crew(agents=[a], tasks=[Task(description='Define latency.', expected_output='One sentence.', agent=a)]).kickoff_async()
u = out.token_usage
print(f'prompt={u.prompt_tokens}  completion={u.completion_tokens}  total={u.total_tokens}  calls={u.successful_requests}')

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Assuming schema descriptions are always read | rubrics ignored on some frameworks/providers | check the prompt, or put rules in the task description |
| Big nested schemas on a small local model | conversion failures | small, flat schemas |
| Treating valid output as correct output | confident wrong categories | test on known cases |
| Hard-coding provider strings everywhere | painful model switches | one helper (`get_llm`) or config |
| Reading `token_usage` across shared LLM objects | inflated costs | fresh LLM per measured run |

## 8 · Check yourself

<details><summary><b>1.</b> How does CrewAI 1.x reach a local Ollama model?</summary>

`LLM(model='ollama/<name>', base_url=...)` uses CrewAI's native OpenAI-compatible client against Ollama. LiteLLM is only needed for providers without a native client.
</details>

<details><summary><b>2.</b> What do <code>output_pydantic</code> and <code>output_json</code> give you?</summary>

A validated Pydantic object in `result.pydantic`, or a dict in `result.json_dict`, for the task's final output.
</details>

<details><summary><b>3.</b> Why did the urgency scale in a field description work here but not in the LangGraph/Agents SDK sections on Ollama?</summary>

CrewAI puts the JSON schema, descriptions included, into the prompt text, so the model reads it. The other two sent the schema as a decoding constraint, which shapes the output without the model reading the descriptions.
</details>

<details><summary><b>4.</b> Where should decision rules go if you want them to work in every framework?</summary>

In the instructions: the task description (or the agent's instructions in other frameworks). The model always reads those.
</details>

## Takeaway

One `LLM` class with `provider/model` names reaches Ollama natively and cloud
providers too. Use `output_pydantic`/`output_json` for results code will act on.
In CrewAI the schema (with its descriptions) is written into the prompt, so the
model reads it, but that's a framework detail, not a law. When rules matter,
check the prompt or state them in the task description.

Next → `04_tools.ipynb`: giving agents tools.